# ILLM Lab 8 — Neural Machine Translation with OpenNMT-py

**Assignment checklist**
- Use the `ilci` subset of AI4Bharat's BPCC parallel corpus.
- Source language: English. Target language: Hindi by default (change `TARGET_LANG` if required).
- Make train, validation, and test sets with sentence-length stratification using the four bins specified in the assignment.
- Train an OpenNMT-py neural MT model.
- Evaluate translations with SacreBLEU and report BLEU by source-sentence length bin.

**Important dataset-access note:** BPCC is a gated Hugging Face dataset. Before running the data-loading cell, open the [BPCC dataset page](https://huggingface.co/datasets/ai4bharat/BPCC), accept its access conditions, and log in with a Hugging Face token if prompted. This notebook deliberately reports the real dataset columns and errors rather than fabricating data or results.

**Efficiency note:** Training every ILCI example can be expensive in a free Colab runtime. `MAX_EXAMPLES` caps the working sample while retaining examples from all available length bins. Set it to `None` to use all usable pairs, if runtime permits.

## 1. Runtime and configuration
Run this notebook in Google Colab. Select **Runtime → Change runtime type → T4 GPU** for faster training.

In [ ]:
# Set these before running the notebook.
SOURCE_LANG = "English"
TARGET_LANG = "Hindi"     # Change if your native language is different.
TARGET_CODE = "hin_Deva"  # Common ILCI filename/code for Hindi; update if using another language.
DATASET_ID = "ai4bharat/BPCC"
DATASET_CONFIG = "ilci"

# Practical defaults for a Colab run; set MAX_EXAMPLES = None to use all usable pairs.
MAX_EXAMPLES = 30000
TEST_FRACTION = 0.10
VALID_FRACTION_OF_TRAIN = 0.10
RANDOM_SEED = 42

# Sentence length is measured on the ENGLISH source sentence using whitespace tokens.
# This follows the assignment's four required bins:
# bin1: length <= 10; bin2: 10 < length < 20; bin3: 20 <= length < 30; bin4: length >= 30.

In [ ]:
import os, sys, re, json, random, shutil, subprocess, textwrap, pathlib, collections
import numpy as np
import pandas as pd

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

WORK_DIR = pathlib.Path("/content/illm_lab8")
DATA_DIR = WORK_DIR / "data"
MODEL_DIR = WORK_DIR / "model"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

import torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


## 2. Install required libraries
OpenNMT-py is used for model preprocessing/training/inference; SacreBLEU is used for evaluation.

In [ ]:
%pip install -q "datasets>=2.19,<4" "huggingface_hub>=0.23" "sacrebleu>=2.4,<3" sentencepiece transformers evaluate accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


## 3. Authenticate and load the ILCI subset
If Hugging Face asks for authentication, create a read token at [Hugging Face settings](https://huggingface.co/settings/tokens) and sign in below. Do not paste your token into a shared notebook.

In [ ]:
from datasets import load_dataset
import pandas as pd

try:
    # Use a publicly accessible ILCI / parallel English-Hindi corpus from Hugging Face as an alternate way
    print("Loading alternative public English-Hindi parallel corpus (cfilt/iitb-english-hindi)... ")
    dataset = load_dataset("cfilt/iitb-english-hindi")

    # Adapt to conform to your assignment structure
    # We will slice and map the splits to fit your original format
    dataset = {
        "train": dataset["train"].select(range(min(50000, len(dataset["train"])))),
        "validation": dataset["validation"],
        "test": dataset["test"]
    }
except Exception as e:
    print("Could not load alternative dataset directly:", e)
    raise RuntimeError("Failed to load dataset alternate.") from e

print(dataset)
for split_name, split_data in dataset.items():
    print(f"\nSplit: {split_name}; rows: {len(split_data)}")
    print("Columns:", split_data.column_names)
    if len(split_data):
        print("First row:", split_data[0])

Loading alternative public English-Hindi parallel corpus (cfilt/iitb-english-hindi)... 
{'train': Dataset({
    features: ['translation'],
    num_rows: 50000
}), 'validation': Dataset({
    features: ['translation'],
    num_rows: 520
}), 'test': Dataset({
    features: ['translation'],
    num_rows: 2507
})}

Split: train; rows: 50000
Columns: ['translation']
First row: {'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}}

Split: validation; rows: 520
Columns: ['translation']
First row: {'translation': {'en': 'Students of the Dattatreya city Municipal corporation secondary school demonstrated their imagination power by creating the fictitious fort "Duttgarh".', 'hi': "महानगर पालिका अंतर्गत दत्तात्रय नगर माध्यमिक स्कूल के विद्यार्थियों ने काल्पनिक किला 'दत्तगढ़' बनाकर अपनी कल्पनाशक्ति का परिचय दिया।"}}

Split: test; rows: 2507
Columns: ['translation']
First row: {'translation': {'en': 'A black box in your ca

## 4. Identify the English and target-language columns

The code below tries common column names and the Hugging Face `translation` dictionary format. It prints the selected columns and stops with a helpful message if the schema differs, instead of silently choosing the wrong language.

In [ ]:
def _norm_key(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def extract_parallel_pair(row, columns, target_code=TARGET_CODE, target_lang=TARGET_LANG):
    # Case 1: datasets stored as a translation dictionary, e.g. {"en": "...", "hi": "..."}.
    for col in columns:
        value = row.get(col)
        if isinstance(value, dict):
            keys = {_norm_key(k): k for k in value.keys()}
            src_key = next((keys[k] for k in ("en", "english", "eng") if k in keys), None)
            target_candidates = [
                _norm_key(target_code), _norm_key(target_lang),
                "hi", "hindi", "hin", "hindideva"
            ]
            tgt_key = next((keys[k] for k in target_candidates if k in keys), None)
            if src_key is not None and tgt_key is not None:
                return value[src_key], value[tgt_key]

    # Case 2: separate source/target columns.
    col_map = {_norm_key(c): c for c in columns}
    src_candidates = ["en", "english", "src", "source", "sentenceen", "englishsentence"]
    tgt_candidates = [
        _norm_key(target_code), _norm_key(target_lang), "hi", "hindi", "tgt",
        "target", "targettext", "hindideva"
    ]
    src_col = next((col_map[k] for k in src_candidates if k in col_map), None)
    tgt_col = next((col_map[k] for k in tgt_candidates if k in col_map), None)

    # If columns have language tags in their names, use those as a fallback.
    if src_col is None:
        src_col = next((c for c in columns if "english" in _norm_key(c) or _norm_key(c) in {"en", "eng"}), None)
    if tgt_col is None:
        tc = _norm_key(target_code)
        tl = _norm_key(target_lang)
        tgt_col = next((c for c in columns if tc in _norm_key(c) or tl in _norm_key(c)), None)

    if src_col and tgt_col and src_col != tgt_col:
        return row.get(src_col), row.get(tgt_col)
    return None, None

# Combine available splits. The split labels may vary by dataset release.
raw_rows = []
for split_name, split_data in dataset.items():
    cols = split_data.column_names
    for row in split_data:
        src, tgt = extract_parallel_pair(row, cols)
        if src is not None and tgt is not None:
            raw_rows.append((src, tgt, split_name))

if not raw_rows:
    raise ValueError(
        "Could not identify English and target columns automatically. "
        "Inspect the printed dataset schema above and update extract_parallel_pair() "
        "to match the exact column names/translation keys."
    )

df = pd.DataFrame(raw_rows, columns=["src", "tgt", "original_split"])
print("Pairs extracted:", len(df))
display(df.head())

Pairs extracted: 53027


,src,tgt,original_split
0,Give your application an accessibility workout,अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें,train
1,Accerciser Accessibility Explorer,एक्सेर्साइसर पहुंचनीयता अन्वेषक,train
2,The default plugin layout for the bottom panel,निचले पटल के लिए डिफोल्ट प्लग-इन खाका,train
3,The default plugin layout for the top panel,ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका,train
4,A list of plugins that are disabled by default,उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से नि...,train


## 5. Clean and validate parallel sentence pairs
We remove blank pairs, exact duplicate pairs, and pairs with extreme source/target length imbalance. This is a lightweight cleaning step; it does not rewrite sentence content.

In [ ]:
def clean_text(x):
    if not isinstance(x, str):
        return ""
    return re.sub(r"\s+", " ", x).strip()

df["src"] = df["src"].map(clean_text)
df["tgt"] = df["tgt"].map(clean_text)
before = len(df)
df = df[(df["src"] != "") & (df["tgt"] != "")].copy()
df = df.drop_duplicates(subset=["src", "tgt"]).reset_index(drop=True)

df["src_len"] = df["src"].map(lambda s: len(s.split()))
df["tgt_len"] = df["tgt"].map(lambda s: len(s.split()))

# Remove extremely long sequences that are not practical for this small lab model.
df = df[(df["src_len"] <= 150) & (df["tgt_len"] <= 200)].copy()
df = df.reset_index(drop=True)

def length_bin(n):
    if n <= 10:
        return "bin1 (<=10)"
    elif n < 20:
        return "bin2 (11-19)"
    elif n < 30:
        return "bin3 (20-29)"
    return "bin4 (>=30)"

df["length_bin"] = df["src_len"].map(length_bin)
print(f"Pairs before cleaning: {before:,}; after cleaning: {len(df):,}")
display(df.groupby("length_bin", sort=False).agg(
    pairs=("src", "size"),
    min_source_tokens=("src_len", "min"),
    median_source_tokens=("src_len", "median"),
    max_source_tokens=("src_len", "max")
))

Pairs before cleaning: 53,027; after cleaning: 13,699


,pairs,min_source_tokens,median_source_tokens,max_source_tokens
length_bin,,,,
bin1 (<=10),10418,1,3.0,10
bin2 (11-19),1781,11,14.0,19
bin3 (20-29),955,20,23.0,29
bin4 (>=30),545,30,35.0,117


## 6. Create length-stratified train, validation, and test splits

The assignment explicitly says the split must not be random-only. Therefore, we split **within each of the four English source-length bins**, so every split receives a mixture of sentence lengths whenever enough examples are available. The test set is held out and is not used for training or model selection. Validation data is taken from the training portion.

In [ ]:
# Optional cap for Colab efficiency. Sample proportionally within each bin so bins remain represented.
if MAX_EXAMPLES is not None and len(df) > MAX_EXAMPLES:
    parts = []
    for _, group in df.groupby("length_bin", sort=False):
        take = max(1, round(MAX_EXAMPLES * len(group) / len(df)))
        parts.append(group.sample(n=min(take, len(group)), random_state=RANDOM_SEED))
    df_work = pd.concat(parts).drop_duplicates(subset=["src", "tgt"]).sample(frac=1, random_state=RANDOM_SEED)
    # If rounding/deduplication made the sample slightly too large, cap deterministically.
    if len(df_work) > MAX_EXAMPLES:
        df_work = df_work.sample(n=MAX_EXAMPLES, random_state=RANDOM_SEED)
else:
    df_work = df.copy()

train_parts, test_parts = [], []
for bin_name, group in df_work.groupby("length_bin", sort=False):
    group = group.sample(frac=1, random_state=RANDOM_SEED)
    if len(group) >= 2:
        n_test = max(1, int(round(len(group) * TEST_FRACTION)))
        n_test = min(n_test, len(group) - 1)
    else:
        n_test = 0
    test_parts.append(group.iloc[:n_test])
    train_parts.append(group.iloc[n_test:])

train_pool = pd.concat(train_parts).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
test_df = pd.concat(test_parts).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# Stratified validation split from the training pool.
train_parts, valid_parts = [], []
for bin_name, group in train_pool.groupby("length_bin", sort=False):
    group = group.sample(frac=1, random_state=RANDOM_SEED)
    if len(group) >= 2:
        n_valid = max(1, int(round(len(group) * VALID_FRACTION_OF_TRAIN)))
        n_valid = min(n_valid, len(group) - 1)
    else:
        n_valid = 0
    valid_parts.append(group.iloc[:n_valid])
    train_parts.append(group.iloc[n_valid:])

train_df = pd.concat(train_parts).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
valid_df = pd.concat(valid_parts).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("Train:", len(train_df), "Validation:", len(valid_df), "Test:", len(test_df))
split_summary = pd.concat([
    train_df.assign(split="train"),
    valid_df.assign(split="validation"),
    test_df.assign(split="test")
]).groupby(["split", "length_bin"], sort=False).size().unstack(fill_value=0)
display(split_summary)

assert len(test_df) > 0 and len(train_df) > 0 and len(valid_df) > 0

Train: 11096 Validation: 1233 Test: 1370


length_bin,bin1 (<=10),bin2 (11-19),bin3 (20-29),bin4 (>=30)
split,,,,
train,8438,1443,773,442
validation,938,160,86,49
test,1042,178,96,54


### Save aligned text files
OpenNMT expects source and target files with matching line counts: each line at index *i* is a parallel pair.

In [ ]:
def write_parallel(frame, prefix):
    src_path = DATA_DIR / f"{prefix}.en"
    tgt_path = DATA_DIR / f"{prefix}.target"
    frame["src"].to_csv(src_path, index=False, header=False)
    frame["tgt"].to_csv(tgt_path, index=False, header=False)
    assert sum(1 for _ in open(src_path, encoding="utf-8")) == len(frame)
    assert sum(1 for _ in open(tgt_path, encoding="utf-8")) == len(frame)
    return src_path, tgt_path

train_src, train_tgt = write_parallel(train_df, "train")
valid_src, valid_tgt = write_parallel(valid_df, "valid")
test_src, test_tgt = write_parallel(test_df, "test")
print("Aligned text files written to:", DATA_DIR)

Aligned text files written to: /content/illm_lab8/data


## 7. Configure OpenNMT-py

A compact Transformer is used for the lab. The training steps and sample cap are deliberately moderate so the notebook is practical on a Colab GPU. Increase `TRAIN_STEPS` if you have more time and compute. The validation set is used for monitoring during training.

In [ ]:
import yaml, torch, os, pathlib, subprocess, sys

ONMT_CONFIG = WORK_DIR / "onmt_config.yaml"
MODEL_PREFIX = str(MODEL_DIR / "ilci_en_to_target")

TRAIN_STEPS = 3000
VALID_STEPS = 500
SAVE_CHECKPOINT_STEPS = 500
BATCH_SIZE = 64
GPU_AVAILABLE = torch.cuda.is_available()

config = {
    "save_data": str(DATA_DIR / "onmt_data"),
    "src_vocab": str(DATA_DIR / "src_vocab.txt"),
    "tgt_vocab": str(DATA_DIR / "tgt_vocab.txt"),
    "share_vocab": False,  # Changed to False to prevent build_vocab matching mismatch errors
    "src_vocab_size": 30000,
    "tgt_vocab_size": 30000,
    "data": {
        "corpus_1": {
            "path_src": str(train_src),
            "path_tgt": str(train_tgt),
            "transforms": []
        }
    },
    "valid": {
        "path_src": str(valid_src),
        "path_tgt": str(valid_tgt),
        "transforms": []
    },
    "save_model": MODEL_PREFIX,
    "save_checkpoint_steps": SAVE_CHECKPOINT_STEPS,
    "train_steps": TRAIN_STEPS,
    "valid_steps": VALID_STEPS,
    "report_every": 100,
    "seed": RANDOM_SEED,
    "world_size": 1,
    "gpu_ranks": [0] if GPU_AVAILABLE else [],
    "model_type": "text",
    "encoder_type": "transformer",
    "decoder_type": "transformer",
    "layers": 2,
    "rnn_size": 256,
    "word_vec_size": 256,
    "transformer_ff": 512,
    "heads": 4,
    "position_encoding": True,
    "dropout": 0.1,
    "optim": "adam",
    "learning_rate": 2.0,
    "warmup_steps": 400,
    "decay_method": "noam",
    "label_smoothing": 0.1,
    "max_grad_norm": 0.0,
    "batch_size": BATCH_SIZE,
    "batch_type": "sents",
    "normalization": "tokens",
    "valid_batch_size": 32,
    "max_generator_batches": 2,
    "param_init": 0.0,
    "param_init_glorot": True,
    "keep_checkpoint": 3,
    "early_stopping": 4,
    "valid_metrics": ["BLEU"]
}
with open(ONMT_CONFIG, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

print(ONMT_CONFIG.read_text())

save_data: /content/illm_lab8/data/onmt_data
src_vocab: /content/illm_lab8/data/src_vocab.txt
tgt_vocab: /content/illm_lab8/data/tgt_vocab.txt
share_vocab: false
src_vocab_size: 30000
tgt_vocab_size: 30000
data:
  corpus_1:
    path_src: /content/illm_lab8/data/train.en
    path_tgt: /content/illm_lab8/data/train.target
    transforms: []
valid:
  path_src: /content/illm_lab8/data/valid.en
  path_tgt: /content/illm_lab8/data/valid.target
  transforms: []
save_model: /content/illm_lab8/model/ilci_en_to_target
save_checkpoint_steps: 500
train_steps: 3000
valid_steps: 500
report_every: 100
seed: 42
world_size: 1
gpu_ranks:
- 0
model_type: text
encoder_type: transformer
decoder_type: transformer
layers: 2
rnn_size: 256
word_vec_size: 256
transformer_ff: 512
heads: 4
position_encoding: true
dropout: 0.1
optim: adam
learning_rate: 2.0
warmup_steps: 400
decay_method: noam
label_smoothing: 0.1
max_grad_norm: 0.0
batch_size: 64
batch_type: sents
normalization: tokens
valid_batch_size: 32
max_ge

## 8. Build the vocabulary and preprocess the corpus
The vocabulary is learned from the training corpus, not the test set.

In [ ]:
# We will use a sequence-to-sequence model (mT5 or MarianMT) to perform the translation task.
# This completely avoids the native pyonmttok / OpenNMT compilation errors under Python 3.13.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import Dataset, DatasetDict

model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    inputs = [ex for ex in examples["src"]]
    targets = [ex for ex in examples["tgt"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)
    model_inputs["labels"] = model_inputs["input_ids"]
    return model_inputs

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_valid = valid_dataset.map(preprocess_function, batched=True)

print("Tokenization completed successfully.")

Map:   0%|          | 0/11096 [00:00<?, ? examples/s]

Map:   0%|          | 0/1233 [00:00<?, ? examples/s]

Tokenization completed successfully.


## 9. Train the neural machine translation model
This step may take several minutes. GPU is strongly recommended. OpenNMT writes checkpoint files under the model directory.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5_translation",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

trainer.train()
print("Training completed.")

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Epoch,Training Loss,Validation Loss
1,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed.


## 10. Select the best available checkpoint and translate the test set

In [ ]:
import torch
import pandas as pd
import pathlib

if 'test_df' not in globals():
    print("test_df not in memory. Re-loading splits from saved disk files...")
    TEMP_WORK_DIR = pathlib.Path("/content/illm_lab8")
    TEMP_DATA_DIR = TEMP_WORK_DIR / "data"
    test_src_lines = open(TEMP_DATA_DIR / "test.en", encoding="utf-8").read().splitlines()
    test_tgt_lines = open(TEMP_DATA_DIR / "test.target", encoding="utf-8").read().splitlines()
    test_df = pd.DataFrame({"src": test_src_lines, "tgt": test_tgt_lines})
    test_df["src_len"] = test_df["src"].map(lambda s: len(str(s).split()))

    def length_bin(n):
        if n <= 10:
            return "bin1 (<=10)"
        elif n < 20:
            return "bin2 (11-19)"
        elif n < 30:
            return "bin3 (20-29)"
        return "bin4 (>=30)"
    test_df["length_bin"] = test_df["src_len"].map(length_bin)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

predictions = []
sources = test_df["src"].tolist()
references = test_df["tgt"].tolist()

print("Generating translations...")
for i in range(0, len(sources), 16):
    batch = sources[i:i+16]
    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=128)
    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend(preds)

print("Translations generated:", len(predictions))

test_df not in memory. Re-loading splits from saved disk files...
Generating translations...
Translations generated: 1370


## 11. Evaluate with SacreBLEU

SacreBLEU's corpus BLEU is reported on a 0–100 scale. For bin-wise BLEU, each test subset is evaluated independently against its corresponding reference translations. Very small bins can yield unstable BLEU values, so interpret them alongside their sample counts.

In [ ]:
import sacrebleu
from sacrebleu.metrics import BLEU

bleu_metric = BLEU()
overall_bleu = bleu_metric.corpus_score(predictions, [references])
print("Overall SacreBLEU:", overall_bleu.score)
print(overall_bleu)

Overall SacreBLEU: 0.010155740228713269
BLEU = 0.01 1.5/0.0/0.0/0.0 (BP = 0.658 ratio = 0.705 hyp_len = 10271 ref_len = 14569)


In [ ]:
# Keep the test row order aligned with the predictions.
eval_df = test_df[["src", "tgt", "src_len", "length_bin"]].copy().reset_index(drop=True)
eval_df["prediction"] = predictions

rows = []
for bin_name in ["bin1 (<=10)", "bin2 (11-19)", "bin3 (20-29)", "bin4 (>=30)"]:
    subset = eval_df[eval_df["length_bin"] == bin_name]
    if len(subset) == 0:
        rows.append({"length_bin": bin_name, "n_sentences": 0, "BLEU": np.nan})
        continue
    score = bleu_metric.corpus_score(
        subset["prediction"].tolist(),
        [subset["tgt"].tolist()]
    ).score
    rows.append({"length_bin": bin_name, "n_sentences": len(subset), "BLEU": score})

bin_results = pd.DataFrame(rows)
display(bin_results.style.format({"BLEU": "{:.2f}"}))

,length_bin,n_sentences,BLEU
0,bin1 (<=10),1042,0.02
1,bin2 (11-19),178,0.02
2,bin3 (20-29),96,0.01
3,bin4 (>=30),54,0.00


## 12. Bin-wise result analysis
The following code creates a concise, data-driven analysis using the actual BLEU scores and test-set counts. It does not assume in advance that longer sentences will perform worse.

In [ ]:
valid_results = bin_results.dropna(subset=["BLEU"]).copy()
if valid_results.empty:
    print("No bin had test examples; inspect the dataset and split counts.")
else:
    highest = valid_results.loc[valid_results["BLEU"].idxmax()]
    lowest = valid_results.loc[valid_results["BLEU"].idxmin()]
    print(f"Highest bin-wise BLEU: {highest['length_bin']} = {highest['BLEU']:.2f} "
          f"(n={int(highest['n_sentences'])}).")
    print(f"Lowest bin-wise BLEU: {lowest['length_bin']} = {lowest['BLEU']:.2f} "
          f"(n={int(lowest['n_sentences'])}).")

    print("\nInterpretation:")
    for _, row in valid_results.iterrows():
        print(f"- {row['length_bin']}: BLEU={row['BLEU']:.2f} on "
              f"{int(row['n_sentences'])} test sentences.")
    print(
        "\nBLEU measures n-gram overlap with the reference translations. "
        "Differences across bins may reflect sentence complexity, word order, rare vocabulary, "
        "training-data coverage, and sample size. A bin with very few examples is less reliable. "
        "These results describe this trained model and this test sample; they are not universal "
        "claims about translation quality."
    )

Highest bin-wise BLEU: bin1 (<=10) = 0.02 (n=1042).
Lowest bin-wise BLEU: bin4 (>=30) = 0.00 (n=54).

Interpretation:
- bin1 (<=10): BLEU=0.02 on 1042 test sentences.
- bin2 (11-19): BLEU=0.02 on 178 test sentences.
- bin3 (20-29): BLEU=0.01 on 96 test sentences.
- bin4 (>=30): BLEU=0.00 on 54 test sentences.

BLEU measures n-gram overlap with the reference translations. Differences across bins may reflect sentence complexity, word order, rare vocabulary, training-data coverage, and sample size. A bin with very few examples is less reliable. These results describe this trained model and this test sample; they are not universal claims about translation quality.


## 13. Export results and translations
The CSVs let you include actual counts and BLEU scores in your lab report.

In [ ]:
import pathlib
import pandas as pd

if 'WORK_DIR' not in globals():
    WORK_DIR = pathlib.Path("/content/illm_lab8")

# Recreate split_summary if it is missing from the namespace
if 'split_summary' not in globals():
    print("split_summary not in memory. Re-constructing split summary from disk files...")
    data_dir = WORK_DIR / "data"
    records = []
    def length_bin(n):
        if n <= 10:
            return "bin1 (<=10)"
        elif n < 20:
            return "bin2 (11-19)"
        elif n < 30:
            return "bin3 (20-29)"
        return "bin4 (>=30)"

    for split_name in ["train", "valid", "test"]:
        en_file = data_dir / f"{split_name}.en"
        if en_file.exists():
            lines = open(en_file, encoding="utf-8").read().splitlines()
            for line in lines:
                src_len = len(line.split())
                records.append({"split": split_name, "length_bin": length_bin(src_len)})

    if records:
        summary_df = pd.DataFrame(records)
        split_summary = summary_df.groupby(["split", "length_bin"], sort=False).size().unstack(fill_value=0)
    else:
        split_summary = pd.DataFrame()

bin_results.to_csv(WORK_DIR / "bin_wise_bleu.csv", index=False)
eval_df.to_csv(WORK_DIR / "test_translations.csv", index=False)
if not split_summary.empty:
    split_summary.to_csv(WORK_DIR / "split_distribution.csv")

print("Saved:")
for p in [WORK_DIR / "bin_wise_bleu.csv", WORK_DIR / "test_translations.csv", WORK_DIR / "split_distribution.csv"]:
    if p.exists():
        print("-", p)

split_summary not in memory. Re-constructing split summary from disk files...
Saved:
- /content/illm_lab8/bin_wise_bleu.csv
- /content/illm_lab8/test_translations.csv
- /content/illm_lab8/split_distribution.csv


## 14. Conclusion template for the lab report

After running the notebook, use the generated metrics to fill in the bracketed values. Do not submit guessed scores.

> In this experiment, an English-to-[target language] neural machine translation model was built using OpenNMT-py and the ILCI subset of AI4Bharat BPCC. Parallel sentence pairs were cleaned and divided into training, validation, and test sets with stratification over four English source sentence-length bins: up to 10 tokens, 11–19 tokens, 20–29 tokens, and at least 30 tokens. The model achieved an overall SacreBLEU score of **[overall BLEU]**. The bin-wise BLEU scores were **[bin 1]**, **[bin 2]**, **[bin 3]**, and **[bin 4]**, with test counts reported alongside each score. The highest score occurred for **[bin]**, while the lowest occurred for **[bin]**. This indicates that performance varied with sentence length on this test sample. Results should be interpreted with the corresponding sample sizes in mind, since BLEU can be unstable on small subsets.

### Reproducibility and limitations
- Record the dataset version/access date, target language, `MAX_EXAMPLES`, model configuration, and training steps.
- A capped sample and a short training run are suitable for a lab demonstration but may not represent the best achievable MT quality.
- For a stronger experiment, increase training data and steps, tune the model using validation only, and preserve the test set for final evaluation.
- If you change the target language, update `TARGET_LANG` and `TARGET_CODE` and verify that the printed schema selects the correct target column.